# 07 -- Datasets, Experiments, and Cost

## Why this notebook exists

In **notebook 06** we instrumented an agent with LangSmith tracing and learned to read individual run spans in the UI. That gave us *observability* for a single execution. But observability is reactive -- you see what happened after the fact. What we want next is *proactive quality control*: run the agent against a curated dataset on demand (or in CI), record every input/output/score pair in the platform, and compare results across agent versions without leaving the browser.

LangSmith calls this **datasets + experiments**. This notebook moves the offline eval harness we built in notebooks 03-05 into that platform workflow: we upload our dataset, adapt our graders into LangSmith evaluators, trigger an experiment run with `evaluate(...)`, and read back aggregate scores, latency, and estimated cost.

## What you'll learn

- How to upload an eval dataset to LangSmith using `Client.create_dataset` and `Client.create_examples`.
- How to adapt a grader that follows our `(example, output) -> Score` contract into a LangSmith evaluator using `as_langsmith_evaluator(grader)` -- and why the LangSmith evaluator signature varies by SDK version.
- How to run a platform-native experiment with `evaluate(target, data=..., evaluators=[...])` -- the LangSmith equivalent of notebook 03's `run_eval`.
- How to compare two experiments side-by-side in the LangSmith UI -- the platform equivalent of notebook 04's regression diff.
- Where to find aggregate latency, token counts, and estimated cost in experiment results, and how to reason about the cost of running LLM-judge evaluators at scale.
- The difference between **offline evaluation** (run a fixed dataset on demand or in CI) and **online evaluation** (sample and score live production traffic).

## 1. Setup + Credential Guard

This notebook requires:

1. **`OPENAI_API_KEY`** -- used by the target agent (calls `gpt-4o-mini`) and by the LLM-judge evaluator.
2. **A free LangSmith account** -- sign up at [smith.langchain.com](https://smith.langchain.com) if you haven't already.
3. **`LANGCHAIN_API_KEY`** -- your LangSmith API key. Copy it from *Settings -> API Keys* in the LangSmith UI and export it in your shell or add it to a `.env` file:
   ```
   export LANGCHAIN_API_KEY="ls__..."
   export LANGCHAIN_TRACING_V2="true"
   export OPENAI_API_KEY="sk-..."
   ```

The guard cell below checks both variables and prints setup instructions if either is missing. **Do not continue past this cell until both checks pass.**

In [ ]:
# Install required packages (safe to re-run; pip is idempotent)
%pip install --quiet langsmith langchain langchain-openai openai python-dotenv

In [ ]:
import os
import sys

from dotenv import load_dotenv
load_dotenv()  # loads OPENAI_API_KEY + LANGCHAIN_API_KEY from project .env if present

missing: list[str] = []

if not os.environ.get("OPENAI_API_KEY"):
    missing.append("OPENAI_API_KEY")

if not os.environ.get("LANGCHAIN_API_KEY"):
    missing.append("LANGCHAIN_API_KEY")

if missing:
    print("=" * 60)
    print("MISSING CREDENTIALS -- notebook will not run correctly.")
    print("=" * 60)
    for var in missing:
        print(f"\n  {var} is not set.")
    print("\nSetup instructions:")
    print("  1. OPENAI_API_KEY:    https://platform.openai.com/api-keys")
    print("  2. LANGCHAIN_API_KEY: https://smith.langchain.com  (free account)")
    print("     In the LangSmith UI: Settings -> API Keys -> Create API Key")
    print("\nThen set the env vars and restart this kernel:")
    print("  export OPENAI_API_KEY='sk-...'")
    print("  export LANGCHAIN_API_KEY='ls__...'")
    print("  export LANGCHAIN_TRACING_V2='true'")
    raise EnvironmentError(f"Missing required credentials: {', '.join(missing)}")

# LangSmith tracing (optional for experiments, but good hygiene)
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")

print("Credentials OK.")
# Print only the last 4 chars of each key -- never print full key bytes
print(f"  OPENAI_API_KEY   : ...{os.environ['OPENAI_API_KEY'][-4:]} found")
print(f"  LANGCHAIN_API_KEY: ...{os.environ['LANGCHAIN_API_KEY'][-4:]} found")
print(f"  LANGCHAIN_TRACING_V2: {os.environ['LANGCHAIN_TRACING_V2']}")

## 2. Re-declare the Minimal Harness

This notebook is self-contained -- it re-declares the four harness pieces it borrows from earlier notebooks so it runs alone without importing from them.

- `Score` and `Example` are the core data types introduced in **notebook 03**.
- `exact_match` is the deterministic grader from **notebook 03**.
- `make_llm_judge(rubric, client, model, key)` is the factory from **notebook 05** that returns a rubric-grading evaluator.

If you have already worked through those notebooks and want to import from them, the signatures are identical -- just replace the cells below with your imports.

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Callable

from openai import OpenAI


# ---------------------------------------------------------------------------
# Core types (from notebook 03)
# ---------------------------------------------------------------------------

@dataclass
class Score:
    """Result of running one grader on one example output."""
    key: str
    score: float          # 0.0 - 1.0
    passed: bool
    comment: str = ""


@dataclass
class Example:
    """One item in an eval dataset."""
    input: Any
    expected: Any = None
    metadata: dict = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Deterministic grader (from notebook 03)
# ---------------------------------------------------------------------------

def exact_match(example: Example, output: str) -> Score:
    """Pass iff the output string equals the expected string exactly."""
    passed = str(output).strip() == str(example.expected).strip()
    return Score(
        key="exact_match",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment="" if passed else f"Expected {example.expected!r}, got {output!r}",
    )


# ---------------------------------------------------------------------------
# LLM-judge factory (from notebook 05 -- hardened version)
# Prompts for JSON {score, passed, rationale}, clamps score to [0,1],
# and wraps JSON parsing in try/except.
# ---------------------------------------------------------------------------

def make_llm_judge(
    rubric: str,
    client: OpenAI,
    model: str = "gpt-4o-mini",
    key: str = "llm_judge",
) -> Callable[[Example, str], Score]:
    """Return a grader that scores an output against *rubric* using an LLM.

    The returned grader follows the same ``(example, output) -> Score``
    contract as all other graders in this series.
    """

    def grader(example: Example, output: str) -> Score:
        system = (
            "You are an impartial evaluator. "
            "Score the assistant output against the rubric. "
            "Reply with JSON only: "
            '{"score": <0.0-1.0>, "passed": <true|false>, "rationale": "<reason>"}'
        )
        user = (
            f"Rubric:\n{rubric}\n\n"
            f"Input:\n{example.input}\n\n"
            f"Expected (if available):\n{example.expected}\n\n"
            f"Output to evaluate:\n{output}"
        )
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            response_format={"type": "json_object"},
            temperature=0,
        )
        try:
            data = json.loads(response.choices[0].message.content)
            raw_score = float(data.get("score", 0.0))
            # Clamp to [0, 1] -- judge may sometimes return values outside range
            clamped_score = max(0.0, min(1.0, raw_score))
            return Score(
                key=key,
                score=clamped_score,
                passed=bool(data.get("passed", False)),
                comment=data.get("rationale", data.get("comment", "")),
            )
        except (json.JSONDecodeError, KeyError, ValueError):
            # Graceful fallback: score 0, flag parsing failure in comment
            return Score(
                key=key,
                score=0.0,
                passed=False,
                comment=f"Judge response parse error: {response.choices[0].message.content[:200]}",
            )

    grader.__name__ = key  # makes the function name match the metric key
    return grader


# ---------------------------------------------------------------------------
# OpenAI client (used by make_llm_judge and the target agents below)
# ---------------------------------------------------------------------------
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
DEFAULT_MODEL = "gpt-4o-mini"

print("Harness re-declared: Score, Example, exact_match, make_llm_judge -- OK")

## 3. Offline vs Online Eval

Before diving into the LangSmith APIs, one conceptual distinction that will frame everything else:

**Offline evaluation** means running a fixed, curated dataset through your agent on demand -- in a notebook like this one, or in CI triggered by a pull request. The dataset is static (or versioned), the inputs are known, and you can compare scores deterministically across agent versions. This is what we built in notebooks 03 and 04, and it is what LangSmith *experiments* are designed for.

**Online evaluation** means sampling real user traffic from your production agent, running graders or LLM judges over those live inputs/outputs, and monitoring quality over time. You can't curate the inputs, so graders need to be more forgiving -- and you typically score only a sample rather than every request.

The rest of this notebook covers **offline evaluation** (the foundation). Notebook 08 sketches how the same suite plugs into an online monitoring loop.

## 4. Upload a Dataset to LangSmith

A LangSmith *dataset* is a versioned collection of (input, expected output) pairs stored in the platform. We create one with `client.create_dataset(...)` and populate it with `client.create_examples(...)`.

We'll use a small capital-city quiz dataset -- five questions with deterministic expected answers -- so `exact_match` can give us reliable signal alongside the LLM judge. The same dataset object will be reused by both experiments in sections 6 and 7.

> **API note (langsmith 0.7.26):** `create_examples` accepts an `examples` list where each element is a dict with `inputs` and `outputs` keys. The older top-level `inputs=`/`outputs=` kwargs still work as legacy fallback but were deprecated in 0.3.11; we use the current form here.

When this cell runs, you'll see the dataset URL printed. Open it to confirm the five rows loaded correctly before running the experiment.

### Try it

In [ ]:
import uuid
from langsmith import Client

ls_client = Client()

# Unique suffix so re-running the notebook doesn't collide with a previous run
_run_id = uuid.uuid4().hex[:8]
DATASET_NAME = f"capital-cities-quiz-{_run_id}"

# Our eval examples: input prompt -> expected answer (exact, trimmed)
_raw_examples = [
    {"input": "What is the capital of France?",        "expected": "Paris"},
    {"input": "What is the capital of Japan?",         "expected": "Tokyo"},
    {"input": "What is the capital of Brazil?",        "expected": "Brasilia"},
    {"input": "What is the capital of Australia?",     "expected": "Canberra"},
    {"input": "What is the capital of South Africa?",  "expected": "Pretoria"},
]

# 1. Create the dataset shell
dataset = ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Capital-city quiz for notebook 07 eval experiments.",
)

# 2. Populate with examples.
#    langsmith 0.3.11+ prefers `examples=[{"inputs": {...}, "outputs": {...}}]`
#    rather than the legacy split `inputs=[...], outputs=[...]` kwargs.
ls_client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs":  {"question": ex["input"]},
            "outputs": {"answer":   ex["expected"]},
        }
        for ex in _raw_examples
    ],
)

dataset_url = f"https://smith.langchain.com/datasets/{dataset.id}"
print(f"Dataset '{DATASET_NAME}' created.")
print(f"  ID  : {dataset.id}")
print(f"  URL : {dataset_url}")
print(f"  Size: {len(_raw_examples)} examples")

## 5. The Evaluator Bridge

LangSmith's `evaluate(...)` function wraps each evaluator callable with `run_evaluator`, which calls `_normalize_evaluator_func` internally (langsmith 0.7.26). That normalizer inspects the function's parameter names: if they include any of `inputs`, `outputs`, `reference_outputs`, `run`, or `example`, it routes the call accordingly. Parameters not in that set fall back to treating them as positional `(run, example)`.

This means our `as_langsmith_evaluator` wrapper can use `inputs`, `outputs`, and `reference_outputs` as **positional parameter names** and LangSmith will populate them automatically -- no need to work with `Run` or `Example` schema objects.

> **Gotcha:** The LangSmith evaluator parameter detection is name-based, not type-based. The supported names in 0.7.26 are: `run`, `example`, `inputs`, `outputs`, `reference_outputs`, `attachments`. Any other parameter name causes the SDK to treat the function as the old `(run, example)` positional form. If you upgrade `langsmith` and get a `TypeError`, check the SDK changelog and verify the supported arg names -- the logic inside the wrapper stays the same.

The return value must be a dict with at minimum `{"key": ..., "score": ...}`. An optional `"comment"` field is also supported and appears in the LangSmith trace view.

### Try it

In [ ]:
def as_langsmith_evaluator(grader: Callable[[Example, str], Score]) -> Callable:
    """Adapt a ``(Example, str) -> Score`` grader into a LangSmith evaluator.

    In langsmith 0.7.26, evaluator functions are wrapped by ``run_evaluator``
    which calls ``_normalize_evaluator_func``. That normalizer detects parameter
    names: if they include ``inputs``, ``outputs``, ``reference_outputs``, etc.,
    it passes the corresponding values as positional or keyword arguments rather
    than the raw ``Run``/``Example`` schema objects.

    We use those named params to bridge to our typed ``(Example, str) -> Score``
    contract:

    - ``inputs``            -- the example's input dict  (keys match ``create_examples``)
    - ``outputs``           -- the target function's return dict
    - ``reference_outputs`` -- the dataset's expected output dict (may be None)

    The return dict ``{"key", "score", "comment"}`` is the shape LangSmith
    expects from a plain-function evaluator.

    Args:
        grader: Any grader following the ``(Example, str) -> Score`` contract.

    Returns:
        A callable with the LangSmith 0.7.26 evaluator keyword-arg signature.
    """
    def _evaluator(
        inputs: dict[str, Any],
        outputs: dict[str, Any],
        reference_outputs: dict[str, Any] | None = None,
        **_kwargs: Any,
    ) -> dict[str, Any]:
        # Build a minimal Example from LangSmith's dicts.
        # Key names must match what we passed to create_examples:
        #   inputs["question"]  and  reference_outputs["answer"]
        question = inputs.get("question", "")
        expected = (reference_outputs or {}).get("answer", None)
        example = Example(input=question, expected=expected)

        # Extract the answer string our target agent returned.
        # The target returns {"answer": <city>}; fall back to "output" if absent.
        output_str = str(outputs.get("answer", outputs.get("output", "")))

        score: Score = grader(example, output_str)
        return {
            "key": score.key,
            "score": score.score,
            "comment": score.comment,
        }

    # Preserve the grader name so it appears correctly in the LangSmith UI
    _evaluator.__name__ = (
        grader.__name__ if hasattr(grader, "__name__") else "grader"
    )
    return _evaluator


# Adapt both graders we'll use in the experiments
ls_exact_match = as_langsmith_evaluator(exact_match)

_quality_rubric = (
    "The answer must name the correct capital city. "
    "Small spelling variants or 'the capital is X' phrasing are acceptable. "
    "Score 1.0 if correct, 0.0 if wrong."
)
_llm_judge_grader = make_llm_judge(
    rubric=_quality_rubric,
    client=openai_client,
    model=DEFAULT_MODEL,
    key="llm_judge_quality",
)
ls_llm_judge = as_langsmith_evaluator(_llm_judge_grader)

print("Evaluator bridge defined: as_langsmith_evaluator -- OK")
print(f"  ls_exact_match : {ls_exact_match.__name__}")
print(f"  ls_llm_judge   : {ls_llm_judge.__name__}")

## 6. Run an Experiment

An *experiment* in LangSmith is one run of a target function over the full dataset, with evaluators scoring each output. The platform records every input, output, and score, and gives you an experiment URL where you can inspect individual rows.

We use `langsmith.evaluate(target, data=..., evaluators=[...])` -- this is the platform-native replacement for notebook 03's hand-rolled `run_eval`. Internally, LangSmith calls our `target` function once per dataset example, collects the outputs, calls each evaluator, and stores everything.

Our target is a small agent that calls `gpt-4o-mini` to answer geography questions. It receives the LangSmith `inputs` dict and must return a dict -- the `"answer"` key is what our evaluators extract.

After this cell finishes (15-30 s), you'll see:
- The experiment name (e.g. `capital-city-v1-20260525T...`)
- A URL to the experiment in the LangSmith UI
- Aggregate scores across all 5 examples

> **Note on DataFrame columns:** `results.to_pandas()` in langsmith 0.7.26 names feedback columns as `feedback.<key>`, not `<key>` directly. So scores appear under `feedback.exact_match` and `feedback.llm_judge_quality`. Latency appears as `execution_time` (seconds).

> **Note on `exact_match`:** It may score below 1.0 even for a correct model because accents (e.g., 'Brasilia' vs 'Brasilia') fail exact string comparison. The LLM judge should score 1.0 for semantically correct answers. This is intentional -- it illustrates why you often want *both* grader types.

### Try it

In [ ]:
from langsmith import evaluate as ls_evaluate


def capital_city_agent_v1(inputs: dict) -> dict:
    """Agent v1: straightforward prompt, returns the city name only."""
    question = inputs["question"]
    response = openai_client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a geography expert. "
                    "Answer with the capital city name only -- no extra words."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    answer = response.choices[0].message.content.strip()
    return {"answer": answer}


print("Running experiment (v1 agent) -- this may take 15-30 s ...")
results_v1 = ls_evaluate(
    capital_city_agent_v1,
    data=DATASET_NAME,
    evaluators=[ls_exact_match, ls_llm_judge],
    experiment_prefix="capital-city-v1",
    metadata={"agent_version": "v1"},
)

print(f"\nExperiment complete.")
print(f"  Experiment name : {results_v1.experiment_name}")

# results_v1.url returns the direct compare URL for this experiment
exp_url_v1 = results_v1.url
print(f"  URL             : {exp_url_v1}")

# Print aggregate scores.
# In langsmith 0.7.26, to_pandas() names feedback columns as "feedback.<key>"
# (e.g. "feedback.exact_match"), not "<key>" directly.
print("\nAggregate scores:")
try:
    df_v1 = results_v1.to_pandas()
    feedback_cols = [c for c in df_v1.columns if c.startswith("feedback.")]
    if feedback_cols:
        for col in feedback_cols:
            metric_name = col.replace("feedback.", "")
            print(f"  {metric_name:<25}: {df_v1[col].mean():.2f}")
    else:
        print("  (No feedback columns found -- check column names:", list(df_v1.columns), ")")
except Exception as exc:
    # Fallback: iterate rows manually if to_pandas() fails
    print(f"  (to_pandas() raised {type(exc).__name__}: {exc} -- iterating rows instead)")
    rows = list(results_v1)
    em_scores = [
        r["evaluation_results"]["results"][0].score
        for r in rows
        if r.get("evaluation_results")
    ]
    if em_scores:
        print(f"  exact_match (first evaluator) mean: {sum(em_scores)/len(em_scores):.2f}")

print(f"\nView in LangSmith UI:")
print(f"  https://smith.langchain.com  -> Experiments -> capital-city-v1-*")

## 7. Compare Experiments Across Agent Versions

In **notebook 04** we diffed two agent versions by comparing score tables in Python. LangSmith gives us the platform-native version of that workflow: run a second experiment against the same dataset and open both experiments in the UI side-by-side.

We'll introduce a deliberately degraded agent (v2) that adds verbose padding to its answers -- causing `exact_match` to drop -- then compare the two experiment URLs.

The LangSmith *Experiments* tab lets you select two (or more) experiments on the same dataset and view a per-row diff: which examples regressed, which improved, and by how much. This is the workflow you'd use in a pull-request review.

After this cell finishes, you'll see both experiment names side-by-side, and a score table showing that `exact_match` dropped for v2 (the preamble breaks exact string comparison) while `llm_judge_quality` stays high (the judge accepts 'The capital city is Paris' as correct).

### Try it

In [ ]:
def capital_city_agent_v2(inputs: dict) -> dict:
    """Agent v2 (deliberately degraded): adds verbose preamble to every answer.

    This causes exact_match to fail even when the city name is correct,
    simulating a prompt-change regression.
    """
    question = inputs["question"]
    response = openai_client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful geography tutor. "
                    "Always begin your answer with 'The capital city is' "
                    "followed by the city name."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    answer = response.choices[0].message.content.strip()
    return {"answer": answer}


print("Running experiment (v2 agent -- degraded) -- this may take 15-30 s ...")
results_v2 = ls_evaluate(
    capital_city_agent_v2,
    data=DATASET_NAME,
    evaluators=[ls_exact_match, ls_llm_judge],
    experiment_prefix="capital-city-v2-degraded",
    metadata={"agent_version": "v2", "note": "verbose preamble regression"},
)

print(f"\nBoth experiments complete.")
print(f"  v1 experiment : {results_v1.experiment_name}")
print(f"  v2 experiment : {results_v2.experiment_name}")


def _mean_score(results, key: str) -> str:
    """Return the mean feedback score for a given metric key, or 'n/a'.

    In langsmith 0.7.26 feedback columns are named 'feedback.<key>'.
    """
    try:
        col = f"feedback.{key}"
        df = results.to_pandas()
        if col in df.columns:
            return f"{df[col].mean():.2f}"
        return "n/a"
    except Exception:
        return "n/a"


print("\nAggregate score comparison:")
print(f"  {'Metric':<25} {'v1':>8} {'v2':>8}")
print(f"  {'-'*25} {'-'*8} {'-'*8}")
for metric in ["exact_match", "llm_judge_quality"]:
    v1_score = _mean_score(results_v1, metric)
    v2_score = _mean_score(results_v2, metric)
    print(f"  {metric:<25} {v1_score:>8} {v2_score:>8}")

print("\nTo compare in LangSmith UI:")
print("  1. Go to smith.langchain.com -> Datasets -> capital-cities-quiz-*")
print("  2. Click 'Compare Experiments' and select both runs.")
print("  3. Inspect the per-row diff: rows where exact_match dropped are the regression.")

## 8. Cost, Latency, and Tokens

LangSmith captures latency and token counts for every traced run. Where the SDK surfaces aggregate cost/latency data on the experiment results object, we can read them directly. For richer breakdowns, the LangSmith UI *Experiments* tab shows per-row and aggregate latency, and the *Trace* view shows token counts per span.

Two things worth internalizing before you run eval at scale:

1. **The cost of the eval itself.** Each example in the dataset triggers one target-agent call *plus* one LLM-judge call. On a 500-example dataset with `gpt-4o-mini`, that is 500 agent calls + 500 judge calls. The judge calls can be as expensive as the agent calls. Prefer deterministic graders where they are sufficient; reserve LLM judges for the dimensions that actually require natural-language reasoning.

2. **Latency budgets for CI.** A 100-example offline suite with a judge takes roughly 2-3 minutes on `gpt-4o-mini`. Plan CI timeouts accordingly, or run the full suite only on main-branch merges and a smaller smoke subset on every PR.

> **Gotcha (langsmith 0.7.26):** `results.to_pandas()` flattens results into a DataFrame with columns `inputs.<key>`, `outputs.<key>`, `reference.<key>`, `feedback.<key>`, `execution_time` (seconds), `error`, `example_id`, and `id`. Token counts are **not** included in the DataFrame -- they live in the LangSmith trace and are visible in the UI's Trace view. If you need token counts programmatically, query them via `ls_client.list_runs(project_name=results_v1.experiment_name)`.

### Try it

In [ ]:
print("=== Cost, Latency, and Token Summary (v1 experiment) ===\n")

try:
    df_v1 = results_v1.to_pandas()
    print("Columns available in results DataFrame:")
    print(" ", list(df_v1.columns))
    print()

    # Latency -- in langsmith 0.7.26 this column is "execution_time" (seconds)
    for lat_col in ["execution_time", "latency", "total_latency"]:
        if lat_col in df_v1.columns:
            print(f"Latency ({lat_col}):")
            print(f"  mean  : {df_v1[lat_col].mean():.2f} s")
            print(f"  median: {df_v1[lat_col].median():.2f} s")
            print(f"  max   : {df_v1[lat_col].max():.2f} s")
            break
    else:
        print("Latency: not available in results DataFrame -- see LangSmith UI trace view.")

    # Token counts are not in the flattened DataFrame in langsmith 0.7.26.
    # They live in the LangSmith trace and can be queried via Client.list_runs.
    print("\nToken counts: not available in to_pandas() output for langsmith 0.7.26.")
    print("  -> Use LangSmith UI: Experiments -> select run -> Trace view for token breakdown.")
    print("  -> Or: ls_client.list_runs(project_name=results_v1.experiment_name)")

except Exception as exc:
    print(f"Note: to_pandas() raised {type(exc).__name__}: {exc}")
    print("This is normal if pandas is not installed or the SDK version differs.")
    print("Use the LangSmith UI: Experiments -> select run -> 'Aggregate' tab.")

print("\n--- Cost reasoning (manual estimate) ---")
_n_examples = len(_raw_examples)
# gpt-4o-mini pricing (illustrative -- check current pricing at openai.com/pricing)
# Roughly $0.15/1M input tokens, $0.60/1M output tokens
_est_tokens_per_call = 150   # rough estimate per agent call
_est_judge_tokens    = 200   # rough estimate per judge call (longer prompt)
_agent_cost  = _n_examples * _est_tokens_per_call * 0.15 / 1_000_000
_judge_cost  = _n_examples * _est_judge_tokens    * 0.15 / 1_000_000
print(f"  Examples in dataset      : {_n_examples}")
print(f"  Estimated agent cost     : ${_agent_cost:.6f}  ({_est_tokens_per_call} tok/call x {_n_examples})")
print(f"  Estimated judge cost     : ${_judge_cost:.6f}  ({_est_judge_tokens} tok/call x {_n_examples})")
print(f"  Combined (illustrative)  : ${_agent_cost + _judge_cost:.6f}")
print()
print("At 500 examples the judge cost alone would be ~100x larger.")
print("Use deterministic graders (exact_match) where they are sufficient.")

## What you just learned

- **LangSmith datasets** are versioned collections of (input, expected output) pairs created with `client.create_dataset(...)` and `client.create_examples(...)`. They are the stable ground truth your offline experiments run against.
- **`as_langsmith_evaluator(grader)`** adapts any grader following our `(Example, str) -> Score` contract into a LangSmith evaluator. The adapter handles the impedance mismatch between LangSmith's `(inputs, outputs, reference_outputs)` keyword-arg signature and our typed dataclass interface -- and is easy to update if the SDK signature changes.
- **`evaluate(target, data=..., evaluators=[...])`** is the platform-native version of notebook 03's hand-rolled `run_eval`. It runs the target function over every dataset example, calls each evaluator, stores results, and returns a results object with `experiment_name`, `url`, and `to_pandas()`.
- **`to_pandas()` in langsmith 0.7.26** produces columns named `feedback.<key>` (e.g. `feedback.exact_match`), `inputs.<key>`, `outputs.<key>`, `reference.<key>`, `execution_time`, `error`, `example_id`, and `id`. Token counts are not included -- use the UI trace view or `Client.list_runs`.
- **Comparing experiments** in the LangSmith UI replicates notebook 04's regression diff visually: select two experiments on the same dataset and inspect per-row score changes.
- **Offline eval** (fixed dataset, run on demand or in CI) is the foundation. **Online eval** (sample live traffic and score it) uses the same graders but on non-curated inputs.
- **LLM-judge calls cost money per example.** Use deterministic graders where they are sufficient and reserve the judge for dimensions that require natural-language reasoning.

## What's missing

We now have every individual component: deterministic graders (02-03), a regression gate (04), an LLM judge (05), tracing (06), and platform datasets + experiments (07). But they still live in separate notebooks. In production you want a *single runnable suite* that wires all of them together into a gate that passes on a good agent and fails on a regressed one -- and that you can invoke from a CI pipeline.

That is exactly what **notebook 08 -- `08_production_eval_gate_end_to_end.ipynb`** builds: a realistic research-and-summarize agent evaluated by a full suite (deterministic graders + regression gate + LLM judge + LangSmith experiment), expressed as a CI-style gate in one runnable cell, with a sketch of the equivalent `pytest` invocation.